In [10]:
 
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
 

In [11]:
import os
os.listdir('/kaggle/input/')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/'

In [ ]:
os.listdir('/kaggle/input/datasets/trishna8/movielens-100k-dataset/ml-100k')

['u.occupation',
 'u1.base',
 'u.info',
 'u4.test',
 'u.item',
 'README',
 'u1.test',
 'ua.test',
 'u.data',
 'u5.test',
 'mku.sh',
 'u5.base',
 'u.user',
 'ub.base',
 'u4.base',
 'u2.test',
 'ua.base',
 'u3.test',
 'u.genre',
 'allbut.pl',
 'u3.base',
 'u2.base',
 'ub.test']

In [ ]:

data_path = "/kaggle/input/datasets/trishna8/movielens-100k-dataset/ml-100k/"

ratings = pd.read_csv(data_path + "u.data", sep="\t",
                      names=["user_id", "movie_id", "rating", "timestamp"])

users = pd.read_csv(data_path + "u.user", sep="|",
                    names=["user_id", "age", "gender", "occupation", "zip"])

movies = pd.read_csv(data_path + "u.item", sep="|",
                     encoding="latin-1", header=None)

movie_columns = [
    "movie_id", "title", "release_date", "video_release_date", "imdb_url",
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]

movies.columns = movie_columns
movies['genres'] = movies.iloc[:, 5:].apply(
    lambda x: ' '.join(x.index[x == 1]), axis=1
)


In [ ]:
# Add ONE synthetic test user with 10 ratings
test_user = pd.DataFrame({
    'user_id': [944] * 10,
    'movie_id': [1, 5, 50, 100, 200, 10, 20, 30, 40, 60],
    'rating': [5, 4, 5, 3, 4, 5, 3, 4, 5, 4],
    'timestamp': [0] * 10
})

ratings = pd.concat([ratings, test_user], ignore_index=True)


In [ ]:
print("="*70)
print("THRESHOLD JUSTIFICATION - STATISTICAL ANALYSIS")
print("="*70)

ratings_per_user = ratings.groupby('user_id').size()
print(f"\nRatings per user statistics:")
print(f"  Mean: {ratings_per_user.mean():.2f}")
print(f"  Median: {ratings_per_user.median():.2f}")
print(f"  Std Dev: {ratings_per_user.std():.2f}")
print(f"  Min: {ratings_per_user.min()}")
print(f"  Max: {ratings_per_user.max()}")

# Calculate quartiles for threshold decisions
q1 = ratings_per_user.quantile(0.25)
q2 = ratings_per_user.quantile(0.50)
q3 = ratings_per_user.quantile(0.75)

print(f"\nQuartile breakdown:")
print(f"  Q1 (25%): {q1:.0f} ratings")
print(f"  Q2 (50%): {q2:.0f} ratings")
print(f"  Q3 (75%): {q3:.0f} ratings")

print(f"\nUsers in each tier:")
cold_start_users = (ratings_per_user < 20).sum()
moderate_users = ((ratings_per_user >= 20) & (ratings_per_user < 70)).sum()
established_users = (ratings_per_user >= 70).sum()

print(f"  < 20 ratings (cold start): {cold_start_users} users ({100*cold_start_users/len(ratings_per_user):.1f}%)")
print(f"  20-70 ratings (moderate): {moderate_users} users ({100*moderate_users/len(ratings_per_user):.1f}%)")
print(f"  > 70 ratings (established): {established_users} users ({100*established_users/len(ratings_per_user):.1f}%)")

print("\n" + "="*70)
print("THRESHOLD RATIONALE:")
print("="*70)
print("""
THRESHOLD 1: < 20 RATINGS (COLD START)
  Reason: Below Q1 (25th percentile). User behavior not yet learnable.
  Action: Use Content-Based (genre similarity) - no user patterns needed.
  Risk: User preferences completely unknown → recommend by features only.

THRESHOLD 2: 20-70 RATINGS (MODERATE)
  Reason: Between Q1-Q3. Enough data for latent factor extraction.
          SVD needs ~15-20 ratings minimum for reliable decomposition.
          70 is safe zone where SVD outperforms simple cosine similarity.
  Action: Use SVD Matrix Factorization - discover hidden preference patterns.
  Benefit: Better than cosine (which still needs ~40+ good neighbors).

THRESHOLD 3: > 70 RATINGS (ESTABLISHED)
  Reason: Above Q3 (75th percentile). Rich user history.
          Enough similar users exist for cosine similarity to find patterns.
          SVD has diminishing returns beyond this point.
  Action: Use Collaborative Filtering (Cosine) - leverage user similarity.
  Benefit: Fastest, proven, finds "people like you" reliably.
""")


THRESHOLD JUSTIFICATION - STATISTICAL ANALYSIS

Ratings per user statistics:
  Mean: 105.94
  Median: 64.50
  Std Dev: 100.93
  Min: 10
  Max: 737

Quartile breakdown:
  Q1 (25%): 33 ratings
  Q2 (50%): 64 ratings
  Q3 (75%): 148 ratings

Users in each tier:
  < 20 ratings (cold start): 1 users (0.1%)
  20-70 ratings (moderate): 494 users (52.3%)
  > 70 ratings (established): 449 users (47.6%)

THRESHOLD RATIONALE:

THRESHOLD 1: < 20 RATINGS (COLD START)
  Reason: Below Q1 (25th percentile). User behavior not yet learnable.
  Action: Use Content-Based (genre similarity) - no user patterns needed.
  Risk: User preferences completely unknown → recommend by features only.

THRESHOLD 2: 20-70 RATINGS (MODERATE)
  Reason: Between Q1-Q3. Enough data for latent factor extraction.
          SVD needs ~15-20 ratings minimum for reliable decomposition.
          70 is safe zone where SVD outperforms simple cosine similarity.
  Action: Use SVD Matrix Factorization - discover hidden preference pat

In [ ]:
# CONTENT BASED FILTERING

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres'])
movie_similarity = cosine_similarity(tfidf_matrix)

def recommend_content_based(user_id, top_n=5):
    liked_movies = ratings[ratings['user_id'] == user_id]
    
    if liked_movies.empty:
        return pd.DataFrame()
    
    # Use top 3 rated movies instead of just 1
    top_movies = liked_movies.sort_values('rating', ascending=False)['movie_id'].head(3).values
    scores = np.mean([movie_similarity[mid - 1] for mid in top_movies], axis=0)
    
    # Exclude movies user already rated
    already_rated = set(liked_movies['movie_id'].values)
    scores_with_ids = [(i + 1, scores[i]) for i in range(len(scores)) if (i + 1) not in already_rated]
    scores_with_ids.sort(key=lambda x: x[1], reverse=True)
    top_movie_ids = [m[0] for m in scores_with_ids[:top_n]]
    
    return movies[movies['movie_id'].isin(top_movie_ids)][["movie_id", "title", "genres"]]

# =

In [ ]:
#SVD MATRIX FACTORIZATION 
user_item_matrix = ratings.pivot(
    index='user_id',
    columns='movie_id',
    values='rating'
).fillna(0)

# Apply SVD decomposition
# n_components=50 captures enough variance without overfitting
# (MovieLens typically uses 20-100 factors; 50 is standard)
svd = TruncatedSVD(n_components=50, random_state=42)
svd_user_factors = svd.fit_transform(user_item_matrix)
svd_movie_factors = svd.components_.T

print(f"\nSVD Decomposition:")
print(f"  User factor matrix shape: {svd_user_factors.shape}")
print(f"  Movie factor matrix shape: {svd_movie_factors.shape}")
print(f"  Explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}")

def recommend_svd(user_id, top_n=5):
    """SVD: Predict ratings using latent factor decomposition"""
    # Get user's latent factors
    user_factors = svd_user_factors[user_id - 1]
    
    # Predict ratings for all movies
    predicted_ratings = np.dot(user_factors, svd_movie_factors.T)
    
    # Get movies user hasn't rated
    user_rated = set(ratings[ratings['user_id'] == user_id]['movie_id'].values)
    
    # Score unrated movies
    recommendations = []
    for movie_id in range(1, len(movies) + 1):
        if movie_id not in user_rated:
            recommendations.append((movie_id, predicted_ratings[movie_id - 1]))
    
    # Sort by predicted rating
    recommendations.sort(key=lambda x: x[1], reverse=True)
    top_movies = [m[0] for m in recommendations[:top_n]]
    
    return movies[movies['movie_id'].isin(top_movies)][["movie_id", "title", "genres"]]



SVD Decomposition:
  User factor matrix shape: (944, 50)
  Movie factor matrix shape: (1682, 50)
  Explained variance ratio: 0.524


In [ ]:
# COLLABORATIVE FILTERING

user_item_normalized = user_item_matrix.copy()
user_means = user_item_matrix.replace(0, np.nan).mean(axis=1)
for user in user_item_normalized.index:
    mask = user_item_normalized.loc[user] > 0
    user_item_normalized.loc[user, mask] -= user_means[user]

user_similarity = cosine_similarity(user_item_normalized)

def predict_rating_collab(user_id, movie_id):
    sim_scores = user_similarity[user_id - 1]
    movie_ratings = user_item_matrix.iloc[:, movie_id - 1]
    
    rated_mask = movie_ratings > 0  
    numerator = np.dot(sim_scores[rated_mask], movie_ratings[rated_mask])
    denominator = np.sum(np.abs(sim_scores[rated_mask]))
    
    return numerator / denominator if denominator > 0 else 0

def recommend_collaborative(user_id, top_n=5):
    """Collaborative: Find movies recommended by similar users"""
    predictions = []
    
    for movie_id in user_item_matrix.columns:
        if user_item_matrix.loc[user_id, movie_id] == 0:
            pred = predict_rating_collab(user_id, movie_id)
            predictions.append((movie_id, pred))
    
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_movies = [m[0] for m in predictions[:top_n]]
    
    return movies[movies['movie_id'].isin(top_movies)][["movie_id", "title", "genres"]]


In [ ]:

def choose_strategy(user_id):
    """
    Select strategy based on user's rating history with clear reasoning.
    
    Returns:
        strategy (str): 'content', 'svd', or 'collaborative'
        num_ratings (int): number of ratings user has
        reason (str): detailed explanation
    """
    num_ratings = ratings[ratings['user_id'] == user_id].shape[0]
    
    if num_ratings < 20:
        strategy = "content"
        reason = (f"Cold Start: User has only {num_ratings} ratings (< 20). "
                 "Insufficient data for pattern discovery. "
                 "Using Content-Based Filtering (genre similarity).")
    
    elif num_ratings < 70:
        strategy = "svd"
        reason = (f"Moderate History: User has {num_ratings} ratings (20-70). "
                 "Enough data to discover latent patterns. "
                 "Using SVD Matrix Factorization (hidden preference factors).")
    
    else:
        strategy = "collaborative"
        reason = (f"Established User: User has {num_ratings} ratings (> 70). "
                 "Rich history enables user similarity matching. "
                 "Using Collaborative Filtering (find similar users).")
    
    return strategy, num_ratings, reason



In [ ]:

def recommend(user_id, top_n=5):
    """
    Generate recommendations with strategy explanation.
    
    Args:
        user_id: User to recommend for
        top_n: Number of recommendations
    
    Returns:
        recommendations: DataFrame with recommended movies
        strategy_info: Dictionary with strategy details
    """
    if user_id not in ratings['user_id'].values:
        print(f"User {user_id} not in dataset. Applying cold start.")
        popular = handle_cold_start_new_user()
        return popular, {"strategy": "cold_start_popular", "num_ratings": 0,
                         "reason": "New user — recommending popular movies."}
    strategy, num_ratings, reason = choose_strategy(user_id)
    
    if strategy == "content":
        recommendations = recommend_content_based(user_id, top_n)
    elif strategy == "svd":
        recommendations = recommend_svd(user_id, top_n)
    else:  # collaborative
        recommendations = recommend_collaborative(user_id, top_n)
    
    strategy_info = {
        "strategy": strategy,
        "num_ratings": num_ratings,
        "reason": reason
    }
    
    return recommendations, strategy_info


In [ ]:

# FULL EVALUATION METRIC
def create_train_test_split(ratings_df, test_size=0.2, random_state=42):
    """Split ratings into train/test for evaluation"""
    train_ratings, test_ratings = train_test_split(
        ratings_df, test_size=test_size, random_state=random_state
    )
    return train_ratings, test_ratings

def evaluate_svd(train_ratings, test_ratings):
    """Evaluate SVD predictions using RMSE and MAE"""
    # Rebuild SVD on training data only
    train_matrix = train_ratings.pivot(
        index='user_id',
        columns='movie_id',
        values='rating'
    ).fillna(0)
    
    svd_eval = TruncatedSVD(n_components=50, random_state=42)
    svd_factors_train = svd_eval.fit_transform(train_matrix)
    svd_factors_movies_train = svd_eval.components_.T
    
    # Make predictions on test set
    actual_ratings = []
    predicted_ratings = []
    
    for _, row in test_ratings.iterrows():
        user_id = int(row['user_id'])
        movie_id = int(row['movie_id'])
        actual_rating = row['rating']
        
        if user_id <= len(svd_factors_train) and movie_id <= len(svd_factors_movies_train):
            user_factors = svd_factors_train[user_id - 1]
            pred_rating = np.dot(user_factors, svd_factors_movies_train[movie_id - 1])
            
            actual_ratings.append(actual_rating)
            predicted_ratings.append(pred_rating)
    
    rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
    mae = mean_absolute_error(actual_ratings, predicted_ratings)
    
    return rmse, mae

def evaluate_collaborative(train_ratings, test_ratings):
    train_matrix = train_ratings.pivot(
        index='user_id', columns='movie_id', values='rating'
    ).fillna(0)
    
    # mean-center before similarity
    train_normalized = train_matrix.copy()
    train_means = train_matrix.replace(0, np.nan).mean(axis=1)
    for user in train_normalized.index:
        mask = train_normalized.loc[user] > 0
        train_normalized.loc[user, mask] -= train_means[user]
    
    user_sim = cosine_similarity(train_normalized)
    
    actual_ratings, predicted_ratings = [], []
    
    for _, row in test_ratings.iterrows():
        user_id = int(row['user_id'])
        movie_id = int(row['movie_id'])
        actual_rating = row['rating']
        
        if user_id <= len(train_matrix) and movie_id <= len(train_matrix.columns):
            sim_scores = user_sim[user_id - 1]
            movie_ratings = train_matrix.iloc[:, movie_id - 1].values
            rated_mask = movie_ratings > 0
            numerator = np.dot(sim_scores[rated_mask], movie_ratings[rated_mask])
            denominator = np.sum(np.abs(sim_scores[rated_mask]))
            pred_rating = numerator / denominator if denominator > 0 else 0
            actual_ratings.append(actual_rating)
            predicted_ratings.append(pred_rating)
    
    rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
    mae = mean_absolute_error(actual_ratings, predicted_ratings)
    return rmse, mae

In [ ]:
def handle_cold_start_new_user(preferred_genres=None):
    """
    For completely new users not in the dataset.
    If genres provided, returns genre-filtered popular movies.
    Otherwise returns overall most popular movies.
    """
    movie_popularity = ratings.groupby('movie_id').size().reset_index(name='num_ratings')
    top_popular = movie_popularity.nlargest(20, 'num_ratings')
    popular_movies = movies[movies['movie_id'].isin(top_popular['movie_id'])][['movie_id', 'title', 'genres']]

    if preferred_genres:
        # Filter to movies matching any preferred genre
        mask = popular_movies['genres'].apply(
            lambda g: any(genre.lower() in g.lower() for genre in preferred_genres)
        )
        filtered = popular_movies[mask]
        return filtered.head(5) if not filtered.empty else popular_movies.head(5)

    return popular_movies.head(5)

In [ ]:

print("\n" + "="*70)
print("TESTING RECOMMENDATIONS")
print("="*70)

# Test with different users
test_users = [5, 19, 50, 100, 200]

for user_id in test_users:
    print(f"\n--- User {user_id} ---")
    recommendations, strategy_info = recommend(user_id, top_n=5)
    
    print(f"Strategy: {strategy_info['strategy'].upper()}")
    print(f"User Ratings: {strategy_info['num_ratings']}")
    print(f"Reason: {strategy_info['reason']}")
    print(f"\nTop 5 Recommendations:")
    print(recommendations[['title', 'genres']])

print("\n" + "="*70)
print("EVALUATION METRICS")
print("="*70)

# Split data
train_ratings, test_ratings = create_train_test_split(ratings)

print("\nTraining SVD...")
svd_rmse, svd_mae = evaluate_svd(train_ratings, test_ratings)

print("\nTraining Collaborative Filtering...")
collab_rmse, collab_mae = evaluate_collaborative(train_ratings, test_ratings)

print("\nResults:")
print(f"\nSVD Matrix Factorization:")
print(f"  RMSE: {svd_rmse:.4f} (lower is better)")
print(f"  MAE:  {svd_mae:.4f} (lower is better)")

print(f"\nCollaborative Filtering (Cosine):")
print(f"  RMSE: {collab_rmse:.4f}")
print(f"  MAE:  {collab_mae:.4f}")

print(f"\nBetter Model: {'SVD' if svd_rmse < collab_rmse else 'Collaborative'}")

# HANDLING COLD START 
print("\n" + "="*70)
print("COLD START DEMO - NEW USER (not in dataset)")
print("="*70)

# Demo 1: No preferences known - return popular movies
print("\nNew user with no preferences:")
print(handle_cold_start_new_user())

# Demo 2: User says they like Action and Thriller
print("\nNew user who likes Action and Thriller:")
print(handle_cold_start_new_user(preferred_genres=["Action", "Thriller"]))




TESTING RECOMMENDATIONS

--- User 5 ---


NameError: name 'ratings' is not defined

In [ ]:
### TESTING COLD START 
recommendations, info = recommend(9999)
print(info['reason'])
print(recommendations[['title', 'genres']])

User 9999 not in dataset. Applying cold start.
New user — recommending popular movies.
                               title                               genres
0                   Toy Story (1995)            Animation Children Comedy
6              Twelve Monkeys (1995)                         Drama Sci-Fi
49                  Star Wars (1977)  Action Adventure Romance Sci-Fi War
55               Pulp Fiction (1994)                          Crime Drama
97  Silence of the Lambs, The (1991)                       Drama Thriller


In [ ]:

# Now test content-based with this user
print("\n" + "="*70)
print("TESTING CONTENT-BASED WITH USER 944 (10 RATINGS)")
print("="*70)

rec = recommend_content_based(944, top_n=5)
print(rec[['title', 'genres']])


TESTING CONTENT-BASED WITH USER 944 (10 RATINGS)
                              title                             genres
12          Mighty Aphrodite (1995)                             Comedy
24             Birdcage, The (1996)                             Comedy
25    Brothers McMullen, The (1995)                             Comedy
40             Billy Madison (1995)                             Comedy
1218          Goofy Movie, A (1995)  Animation Children Comedy Romance
